## Imports

<a href="https://colab.research.google.com/github/sylvainestebe/european-city-inference/blob/main/notebooks/tutorial_1_decision_making.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import jax
import matplotlib.pyplot as plt
import pandas as pd

from eci.decision import response_function
from eci.environment import EnvConfig, Environment
from eci.metrics import batch_compute_metrics
from eci.plots import plot_preference, plot_voting_metrics
from eci.utils import _extract_env_data_vectorized
from eci.voting import _vote_plurality, _vote_quadratic

## Parameter Configuration 
The simulation constants are defined here.

In [ ]:
config = EnvConfig(
    num_voters=100,
    num_candidates=5,
    num_preferences=4,
)
NUM_SIMULATIONS = 100_000  # Number of simulation steps

## Environment Initialization 
The environment is created using the parameters defined above.

In [ ]:
# Initialize environment
env = Environment(config)
env.num_simulations = NUM_SIMULATIONS
env._run_multi_agent_inference()

In [ ]:
data = _extract_env_data_vectorized(env)

In [ ]:
fig, ax = plot_preference(data)
fig.show()

## Voting systems
Now we will explore the different voting system implemented.

### Plurality Voting

Each voter casts **one vote** for the candidate they prefer the most. The candidate with the most votes wins (a single round is used here).

**Algorithm**

1. For each voter $i$, the [response function](https://centre-for-humanities-computing.github.io/european-city-inference/extending_response_functions/) computes a utility score $u_{i,c}$ for every candidate $c$.
2. A probability of voting for each candidate is obtained via softmax over those scores:
$$ p_{i,c} = \frac{\exp(u_{i,c})}{\sum_{c'} \exp(u_{i,c'})} $$
1. Each voter samples **one** vote $v_i$ from the categorical distribution $\mathrm{Cat}(p_{i,:})$.
2. The winner is the candidate with the most votes:
$$ \mathrm{winner} = \arg\max_c \;\#\{i : v_i = c\} $$

**Properties.** Plurality records *only the direction* of each voter's preference — the intensity is discarded. Two voters who barely prefer C1 over C2 can cancel out a voter who strongly prefers C2. Ties are broken by the lowest candidate index.

In [ ]:
data = _extract_env_data_vectorized(env)
key = jax.random.PRNGKey(42)
# Run simulations
sim_plurality = env.run_n_simulation(
    func=_vote_plurality,
    data=data,
    response_function=response_function,
    key=key,
    n_simulations=NUM_SIMULATIONS,
)
# Compute metrics
metrics_plurality = batch_compute_metrics(sim_plurality)

### Quadratic Voting

Each voter has a fixed **budget** $B$ of credits and distributes them across candidates. The cost of casting $v$ votes for a candidate is $v^2$, so doubling the votes costs four times as much. This penalises concentration and lets voters express **how much** they prefer a candidate, not just **which** one.

**Algorithm** (heuristic implementation in `_compute_sequential_qv_allocation`)

1. Utilities $u_{i,c}$ from the response function (same as plurality).
2. **Pick top-$k$ candidates** per voter via Gumbel-top-$k$ — equivalent to sampling $k$ distinct candidates without replacement with probability $\propto \mathrm{softmax}(u_{i,:})$. Default $k = 5$.
3. **Allocate credits** per pick proportional to softmax of utilities, scaled to the per-pick share of the budget:
$$ c_{i,c} = \mathbb{1}[c \in \mathrm{top\text{-}k}_i] \cdot \big(\mathrm{softmax}(u_{i,c}) \cdot \tfrac{B}{k} + \varepsilon\big),\quad \varepsilon \sim \mathcal{N}(0,\sigma^2) $$
   (small jitter $\varepsilon$ breaks ties when every candidate is picked.)
4. **Convert credits to votes** via the square-root rule:
$$ v_{i,c} = \lfloor \sqrt{c_{i,c}} \rfloor $$
5. The **winner** is the candidate with the most total votes:
$$ \mathrm{winner} = \arg\max_c \sum_i v_{i,c} $$

In [ ]:
# Run simulations
sim_qv = env.run_n_simulation(
    _vote_quadratic,
    data=data,
    response_function=response_function,
    key=key,
    n_simulations=NUM_SIMULATIONS,
)

# Compute metrics
metrics_qv = batch_compute_metrics(sim_qv)

# Metrics

For every simulation we read out two metrics.

**Winner satisfaction** — the electorate's aggregate utility for the elected
candidate:
$$W_s = \sum_i u_{i,\,c^{*}}.$$
Higher means the winner is collectively preferred. It depends only on *who*
wins, so it takes one value per possible winner.

**Vote efficiency** — how well each agent's votes land on the candidates it
actually values, summed over the electorate:
$$V_e = \sum_i \frac{\sum_c v_{i,c}\,u_{i,c}}{\sum_c v_{i,c}}.$$
Higher means votes are "well spent". Unlike $W_s$, it depends on *how* the
ballot is cast — so plurality (one vote) and quadratic voting (a spread of
credits) can differ even for the same winner.

**Reading the plot.** Each point is one simulation — one stochastic election
run. The vertical spread of points reflects the run-to-run variability of
vote sampling; the offset between the plurality and quadratic clouds shows
which rule transmits voter preferences more faithfully.

In [ ]:
# change voting system labels
metrics_plurality["voting_system"] = "Plurality"
# metrics_plurality_strat["voting_system"] = "Plurality_Strat"
metrics_qv["voting_system"] = "Quadratic"
# metrics_qv_strat["voting_system"] = "Quadrati_Strat"

# combine dataframes
combined_df = pd.concat(
    [
        metrics_plurality,
        metrics_qv,
    ],
    ignore_index=True,
)

# plot voting metrics
fig = plot_voting_metrics(combined_df)
plt.show()